# DE

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import random
from typing import List, Tuple

import numpy as np
from Helper_invert_1d import initialize_inversion, fitness_s, unpack_qpso_solutions


def invert_first_output_de_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model_path: str,
    x_fixed_raw: np.ndarray,         # fixed inputs, raw scale (len 10)
    actual_x_adj_raw: np.ndarray,    # actual adjustable inputs, raw scale (len 4)
    target_raw: float,               # desired raw value of first output
    pop_size: int,                   # DE population size
    num_generations: int,            # number of DE generations
    F: float,                        # mutation factor
    CR: float,                       # crossover probability
    hidden_dims: List[int],          # network hidden-layer sizes
    activation: str,                 # activation name used in training
    x_adj_intervals_raw: np.ndarray, # raw [min,max] for each adjustable input (4×2)
    Q_bounds: Tuple[float, float],    # raw bounds (Q_min, Q_max)
    k: int                           # number of top unique solutions to return
) -> List[Tuple[np.ndarray, np.ndarray, float]]:
    """
    Differential-Evolution inversion of the first network output under
    the constraint x1_raw = Q * x0_raw. Returns up to k+1 results:
      (adjustable_inputs_raw, all_outputs_raw, residual_raw),
    where the first tuple is the actual input→output from data.
    """

    # -- [1] Initialize everything (scalers, model, actual result, bounds) ----
    init = initialize_inversion(
        x_scaler_path, y_scaler_path, model_path,
        hidden_dims, activation,
        x_fixed_raw, actual_x_adj_raw,
        target_raw, x_adj_intervals_raw, Q_bounds
    )
    model         = init["model"]
    x_fixed_s     = init["x_fixed_s"]
    means_x       = init["means_x"]
    scales_x      = init["scales_x"]
    y_scaler      = init["y_scaler"]
    results       = init["results"]       # starts with (actual_x_adj_raw, preds_act_raw, resid_act)
    min_vec       = init["min_vec"]       # lower bounds for [x0_s, x2_s, x3_s, Q]
    max_vec       = init["max_vec"]       # upper bounds
    target_scaled = init["target_scaled"] # scaled target for fitness

    # -- [2] Initialize DE population -----------------------------------------
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]
    fitnesses  = [
        fitness_s(ind, model, x_fixed_s, means_x, scales_x, target_scaled)
        for ind in population
    ]

    # -- [3] Differential Evolution loop --------------------------------------
    dim = len(min_vec)
    for _ in range(num_generations):
        for i in range(pop_size):
            # pick three distinct others
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c = random.sample(idxs, 3)
            x_a, x_b, x_c = population[a], population[b], population[c]

            # mutation: v = x_a + F * (x_b - x_c)
            v = x_a + F * (x_b - x_c)
            v = np.clip(v, min_vec, max_vec)

            # crossover → trial vector
            jrand = random.randrange(dim)
            trial = np.array([
                v[j] if (random.random() < CR or j == jrand) else population[i][j]
                for j in range(dim)
            ])
            trial = np.clip(trial, min_vec, max_vec)

            # selection
            f_trial = fitness_s(trial, model, x_fixed_s, means_x, scales_x, target_scaled)
            if f_trial < fitnesses[i]:
                population[i] = trial
                fitnesses[i]  = f_trial

    # -- [4] Select top-k unique candidates -----------------------------------
    scored_final = sorted(zip(fitnesses, population), key=lambda t: t[0])
    unique = []; seen = set()
    for _, cand in scored_final:
        key = tuple(np.round(cand, 6))
        if key in seen:
            continue
        seen.add(key)
        unique.append(cand)
        if len(unique) >= k:
            break

    # -- [5] Inverse-scale & compute raw predictions + residuals -------------
    other_results = unpack_qpso_solutions(
        unique,
        means_x,
        scales_x,
        x_fixed_s,
        model,
        y_scaler,
        target_raw
    )
    results.extend(other_results)

    return results
